# Machine Learning. Introducción

## Antes de empezar:
Si te has perdido en la creación de código: https://codeshare.io/

## Imprescindibles en Python para ML
- **pandas** (https://pandas.pydata.org/docs/user_guide/index.html): manipulación y análisis de datos estructurados mediante DataFrames y Series, con soporte para limpieza, transformación y agregación.
- **numpy** (https://numpy.org/doc/stable/user/): cálculo numérico eficiente a través de arrays y funciones matemáticas
- **sklearn** (scikit-learn): https://scikit-learn.org/stable/user_guide.html): todo lo que necesario para iniciarse en Machine Learning, incluyendo preprocesamiento, modelos, selección de características y evaluación
- **matplotlib** (https://matplotlib.org/stable/users/explain/quick_start.html): creación de gráficos, imprescindibles para la evaluación de los modelos

# Algunas librerías avanzadas para ML
- imbalanced data (https://pypi.org/project/imbalanced-learn/): para manejar datos desbalanceados; incluye técnicas como sobremuestreo, submuestreo y generación de datos sintéticos
- seaborn (https://seaborn.pydata.org/tutorial.html): usado con matplotlib para mejorar los gráficcos

In [19]:
# Importar las bibliotecas necesarias
import warnings
warnings.filterwarnings('ignore')  # Ignorar las advertencias para una salida más limpia

import pandas as pd  # Manipulación de datos
import numpy as np  # Operaciones numéricas
import matplotlib.pyplot as plt  # matplotlib.pyplot para gráficos
import seaborn as sns  # seaborn para gráficos estadísticos

sns.set_style('darkgrid')  # Establecer el estilo de los gráficos de seaborn

# Algunas peculiaridades sobre sklearn
Aunque puede importarse el módulo entero, esta librería está dividida en submódulos según su funcionalidad. 
- **Modelos de ML**: ensemble, linear model,
- **Preprocesamiento** (https://scikit-learn.org/1.5/modules/preprocessing.html)  (_.preprocessing_): StandardScaler, LabelEncoder, etc.
- **Estructuras de validación** (https://scikit-learn.org/1.5/api/sklearn.model_selection.html) (_.model_selection_): KFold, train_test_split
- **Métricas de evaluación** (https://scikit-learn.org/1.5/modules/model_evaluation.html) (*.metrics*): accuracy_score, mean_squared_error
- **Datasets de ejemplo** (https://scikit-learn.org/1.5/api/sklearn.datasets.html) (_.datasets _): load_iris, load_digits...

## Ejemplo: 
¿qué necesitamos importar para en un análisis predictivo basado en una regresión sobre el conjunto de datos "wine", normalizar los datos, usar el accuracy como medida de bondad de ajuste en un k-fold estratificado

In [35]:
from sklearn.model_selection import StratifiedKFold #importar el k-fold estratificado
from sklearn.preprocessing import StandardScaler #importar el transformador para normalizar
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score #importar la medida de bondad de ajuste, accuracy
from sklearn.datasets import load_wine #importar la base de datos

# Proceso completo ML
- **Data preparation** (80% tiempo)
  1. Exploring
  2. Cleansing
  3. Transformation
- **Model training** (20% tiempo)
  1. Feature selection: _¿qué variables explicativas voy a usar y por qué?_
  2. Model validation: _¿qué estructura de validación voy a usar?_
  3. Metric analysis: _¿en base a qué métrica quiero analizar mis resultados? ¿me interesa ganar en especificidad? ¿me afectan mucho los falsos negativos?_
  4. Model selection: _¿qué técnica predictiva voy a aplicar?_
  5. Model tuning: _ajuste de hiperparámetros para la técnica escogida_

# IMPORTANTE:
1. Por mucho que ajuste los modelos, **si no** se ha hecho un **correcto preprocesamiento** de los datos es **IMPOSIBLE** obtener buenos resultados (especialmente, a la larga)
2. Cada **técnica de ML** exige sus **propias especificaciones** en el preprocesamiento.


## Controlando la aleatoriedad
Valor numérico que se utiliza para **inicializar el generador de números aleatorios**

Imprescindible para asegurar la **reproducibilidad de los experimentos**

**En ningún caso es un hiperparámetro a tunear**

**OJO**: numpy tiene su propio proceso de aleatorización y control, importante tener claro siempre el proceso!!!!

In [77]:
# Establecer una semilla concreta (en este caso, 12345)
SEED = 12345

# Para aleatorización en Python
import random
random.seed(SEED)

# Para aleatorización en numpy
import numpy as np
np.random.seed(SEED)

# Para aleatorización en scikit-learn (por ejemplo, train_test_split)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=SEED)

# O puedes simplemente usar el valor de SEED cada vez que aparezca un elemento aleatorio


## Ahora sí, empezamos a practicar con ML
1. De forma generalizada, se utiliza _X_ para nombrar el conjunto de variables explicativas, e _y_ para la variable objetivo
2. Asumiendo que ya se ha hecho el preprocesamiento, estructura general:
    1. División de datos (_.model_selection_)
    2. Escoger el modelo
    3. Ajustar el modelo (_.fit_)
    4. Validar el modelo

## Ejemplo 1
Cargar el conjunto de datos _wine_, conocer de qué tratan las variables, aplicar una regresión logística usando una estructura tr/ts con tr=75%, y validar el modelo en base al accuracy

## Ejemplo 2
Aplicar un proceso similar al del Ejemplo 1, pero en este caso, trabajar sobre datos normalizados

In [2]:
from sklearn.datasets import load_wine
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

seed = 1234
data = load_wine()
X = data.data
y = data.target

X_train,X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=seed)

scaler = StandardScaler()
X_scaled_tr = scaler.fit_transform(X_train)
X_scaled_ts = scaler.fit_transform(X_test)

model = LogisticRegression(random_state=seed)
model.fit(X_scaled_tr, y_train)#entreno

y_pred = model.predict(X_scaled_ts)#predigo

cm = confusion_matrix(y_test,y_pred)
print(cm)

[[14  0  0]
 [ 0 18  1]
 [ 0  0 12]]


## OJO!!
¿Encuentras alguna diferencia en el proceso de transformación entre los Ejemplos 1 y 2?
Las transformaciones de datos deben hacerse después de dividir los datos en train y test. 
1. Evitar la contaminación de datos (_data leakage_): el escalador tiene acceso a información de los datos de prueba durante el ajuste, lo que nos puede llevar a modelos no generalizables, ya que los datos de prueba influyen indirectamente en el entrenamiento.
2. Simular un _escenario real_, en el que los datos de prueba nunca estarán disponibles durante el entrenamiento

# Ejemplo 3
Trabajar con un problema de predicción de variable continua, *california_housing*, sobre información de viviendas de California y su precio medio. Trabaja con datos estandarizados y evalúa el modelo de regresión usando el MSE. El conjunto de datos lo tienes disponible en formato _.csv_.

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import pandas as pd
df = pd.read_csv(r"C:\Users\maria\OneDrive\Documentos 1\Escritorio\MASTER\MODULO 7 MACHINE LEARNING\Datasets\california_housing.csv")
df.head()
seed = 1234
X = df.drop('MedHouseVal', axis = 1)
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.7, random_state=seed)

scaler = StandardScaler()

X_scaled_tr = scaler.fit_transform(X_train)
X_scaled_ts = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_scaled_tr, y_train)

y_pred = model.predict(X_scaled_ts)

mse = mean_squared_error(y_test, y_pred)
print(f"mse test: {mse}")

y_pred_tr = model.predict(X_scaled_tr)

mse_tr = mean_squared_error(y_train, y_pred_tr)
print(f"mse train: {mse_tr}")



mse test: 0.5242876152016424
mse train: 0.5302420528369417


# Ejemplo 4
Hacer un análisis predictivo del conjunto de datos _diamonds_ disponible en _sns_. Trabajar la regresión con datos estandarizados, sólo con variables numéricas y evaluar los resultados con R2. Usar una estructura de validación tr/ts dejando el 20% de los datos para la prueba.